<div dir="rtl">

# AI Agents 101: LangChain, LangGraph & MCP

מחברת זו מציגה את שלושת הכלים המרכזיים לבניית סוכני AI מודרניים:

| כלי | תפקיד |
|-----|-------|
| **LangChain** | אבני הבניין — prompts, tools, chains, memory |
| **LangGraph** | מנוע הסוכן — גרף מצבים עם לולאות וסניפים |
| **MCP** | שכבת החיבור — פרוטוקול סטנדרטי לכלים חיצוניים |

---

## למה בכלל צריך סוכני AI?

מודל שפה פשוט מקבל טקסט ומחזיר טקסט — זהו. סוכן AI הוא משהו אחר:
- הוא **מחליט** אילו פעולות לבצע
- הוא **מבצע** פעולות (קוראים לזה *tool calls* או *function calling*)
- הוא **מתבונן** בתוצאות וממשיך בהתאם
- הוא **שומר מצב** לאורך ריצות מרובות

```
┌─────────────────────────────────────┐
│           Agent Loop                │
│                                     │
│  Think → Act → Observe → Think...   │
│                                     │
│  (ReAct pattern: Reason + Act)      │
└─────────────────────────────────────┘
```

</div>

<div dir="rtl">

## 1. התקנה

</div>

In [ ]:
# התקנת כל הספריות הדרושות
# MCP client library from Anthropic
%pip install -q langchain langchain-anthropic langgraph langchain-mcp-adapters anthropic

In [ ]:
import os
import getpass

# טען את המפתח — אם כבר קיים בסביבה, השתמש בו; אחרת בקש אותו
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter ANTHROPIC_API_KEY: ")

assert os.environ.get("ANTHROPIC_API_KEY"), "חסר ANTHROPIC_API_KEY!"
print("✓ API key found")

In [ ]:
key_string = 'YOUR_API_KEY_HERE'

<div dir="rtl">

---

# חלק א: LangChain — אבני הבניין

LangChain היא ספריית הבסיס. היא מגדירה את כל הרכיבים הבסיסיים:

```
LangChain
├── ChatModel     — ה-LLM עצמו (Claude, GPT, Gemini...)
├── PromptTemplate — תבניות להנחיות
├── Tool          — פונקציה שהסוכן יכול לקרוא לה
├── Memory        — זיכרון בין פניות
└── Chain         — חיבור רכיבים ברצף
```

### מה LangChain *לא* עושה?
LangChain לא מנהלת **לולאות** ו**מצב** — לזה יועיד LangGraph.

</div>

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage

# יצירת מודל
model = ChatAnthropic(model="claude-haiku-4-5-20251001

# קריאה פשוטה — זה עדיין לא סוכן, זה רק LLM
response = model.invoke([
    SystemMessage(content="You are a helpful assistant. Be concise."),
    HumanMessage(content="What is 2+2?")
])

print(response.content)

<div dir="rtl">

### כלים (Tools) — הלב של הסוכן

כלי הוא פונקציה Python רגילה עם `@tool` decorator. הסוכן מחליט מתי לקרוא לה.

</div>

In [ ]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together."""
    return a * b

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    # סימולציה — בפועל היית קורא ל-API אמיתי
    weather_data = {
        "tel aviv": "25°C, sunny",
        "jerusalem": "18°C, cloudy",
        "haifa": "22°C, partly cloudy",
    }
    return weather_data.get(city.lower(), f"No data for {city}")

tools = [add, multiply, get_weather]

# הצג את ה-schema שהמודל רואה
print("Tool schemas:")
for t in tools:
    print(f"  - {t.name}: {t.description}")
    print(f"    args: {t.args}")

In [ ]:
# חיבור כלים למודל — bind_tools אומר למודל "אלה הכלים שלך"
model_with_tools = model.bind_tools(tools)

# נשאל משהו שדורש כלי
msg = model_with_tools.invoke([HumanMessage(content="What's the weather in Tel Aviv?")])

print(f"Response type: {type(msg).__name__}")
print(f"Content: {msg.content}")
print(f"Tool calls: {msg.tool_calls}")

<div dir="rtl">

**שים לב:** המודל החזיר `tool_calls` — הוא *ביקש* לקרוא לכלי, אבל עדיין לא קרא לו!

זה הבדל חשוב: המודל מחליט `"אני רוצה לקרוא ל-get_weather('Tel Aviv')"` — **מישהו אחר** צריך לבצע את הקריאה ולהחזיר את התוצאה.

זה בדיוק מה ש-**LangGraph** עושה.

</div>

<div dir="rtl">

---

# חלק ב: LangGraph — מנוע הסוכן

LangGraph מגדיר את ה**לוגיקה** של הסוכן: מה קורה אחרי כל שלב?

```
                    ┌──────────────┐
        ┌──────────►│   agent      │
        │           │  (LLM call)  │
        │           └──────┬───────┘
        │                  │
        │        ┌─────────▼──────────┐
        │        │ should_continue?   │
        │        └────┬───────────┬───┘
        │         yes │           │ no
        │    ┌────────▼───────┐   │
        └────│  tools         │   ▼
             │  (execute!)    │  END
             └────────────────┘
```

### מושגי מפתח ב-LangGraph:
- **State** — מילון שעובר בין הצמתים ומצטבר
- **Node** — פונקציה שמקבלת state ומחזירה עדכון
- **Edge** — קשר בין צמתים (יכול להיות מותנה)
- **Graph** — הכל יחד

</div>

In [ ]:
# דוגמה מינימלית של Tree of Thoughts ב-LangGraph
# הגרף: expand → evaluate → prune/continue → תשובה סופית

from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, SystemMessage
import json

class ToTState(TypedDict):
    question: str
    thoughts: list[dict]   # [{"text": ..., "score": ...}]
    best_thought: str
    answer: str

# צומת 1: expand — בקש מה-LLM 3 גישות שונות לפתרון
def expand(state: ToTState):
    prompt = f"""Given this question: "{state['question']}"
Generate exactly 3 different approaches to solve it.
Return a JSON array: [{{"approach": "...", "reasoning": "..."}}]
Return ONLY the JSON array, no other text."""
    
    response = model.invoke([HumanMessage(content=prompt)])
    try:
        thoughts = json.loads(response.content)
    except:
        thoughts = [{"approach": response.content, "reasoning": ""}]
    
    return {"thoughts": [{"text": t["approach"], "reasoning": t.get("reasoning",""), "score": 0} for t in thoughts]}

# צומת 2: evaluate — דרג כל גישה
def evaluate(state: ToTState):
    scored = []
    for thought in state["thoughts"]:
        prompt = f"""Question: "{state['question']}"
Approach: "{thought['text']}"
Rate this approach from 1-10 for correctness and efficiency.
Return ONLY a single integer (1-10)."""
        
        response = model.invoke([HumanMessage(content=prompt)])
        try:
            score = int(response.content.strip())
        except:
            score = 5
        scored.append({**thought, "score": score})
    
    return {"thoughts": scored}

# צומת 3: prune + answer — בחר הכי טוב וענה
def answer(state: ToTState):
    best = max(state["thoughts"], key=lambda t: t["score"])
    
    prompt = f"""Question: "{state['question']}"
Best approach identified: "{best['text']}"
Now provide a complete, concise answer using this approach."""
    
    response = model.invoke([HumanMessage(content=prompt)])
    return {"best_thought": best["text"], "answer": response.content}

# בניית הגרף
tot_builder = StateGraph(ToTState)
tot_builder.add_node("expand", expand)
tot_builder.add_node("evaluate", evaluate)
tot_builder.add_node("answer", answer)

tot_builder.add_edge(START, "expand")
tot_builder.add_edge("expand", "evaluate")
tot_builder.add_edge("evaluate", "answer")
tot_builder.add_edge("answer", END)

tot_graph = tot_builder.compile()

# הרץ
result = tot_graph.invoke({
    "question": "What's the best way to reverse a string in Python?",
    "thoughts": [],
    "best_thought": "",
    "answer": ""
})

print("=== Tree of Thoughts Results ===\n")
print("Approaches explored:")
for i, t in enumerate(result["thoughts"]):
    print(f"  {i+1}. [{t['score']}/10] {t['text']}")

print(f"\nBest approach chosen: {result['best_thought']}")
print(f"\nFinal answer:\n{result['answer']}")

<div dir="rtl">

### הקשר ל-Chain of Thought ו-Tree of Thoughts

**Chain of Thought (CoT)**
הרעיון הבסיסי: לבקש מה-LLM לחשוב בקול לפני שהוא עונה. פשוט מאוד — prompt אחד, תשובה אחת, אבל עם שלבי ביניים.

```
שאלה: "כמה עצים בגינה? יש 3 שורות של 4 עצים, וגם 5 עצים בודדים."
תשובה ללא CoT: 17
תשובה עם CoT: "3×4=12, ועוד 5 בודדים, סה"כ 17"
```

CoT הוא **נתיב לינארי אחד** בגרף — חשיבה → תשובה. מכוון, לא מסתעף.

---

**Tree of Thoughts (ToT)**
ההכללה שציינת: במקום נתיב אחד, מחקרים (Yao et al., 2023) הציעו לפרוס **עץ** של מחשבות — לייצר כמה כיוונים חשיבה במקביל, להעריך כל אחד, ולהמשיך רק את המבטיחים.

```
                  שאלה
                 /  |  \
           רעיון א  ב   ג        ← פיתוח מחשבות מקביל
           /  \      |
         א1   א2     ב1          ← הרחבה של המבטיחים
               |
             פתרון               ← הכי טוב הגיע לכאן
```

זה כבר **גרף** — בפרט עץ. ב-LangGraph מממשים ToT כך:
- צומת **expand**: מבקש מה-LLM k רעיונות שונים
- צומת **evaluate**: מדרג כל רעיון (1-10)
- קשת מותנית **prune**: ממשיך רק עם הרעיונות שמעל סף

| גישה | מבנה הגרף | מספר קריאות LLM |
|------|----------|-----------------|
| **Vanilla** | node יחיד | 1 |
| **CoT** | נתיב לינארי | 1 (prompt ארוך יותר) |
| **ReAct** | לולאה (cycle) | n (עד שמסיים) |
| **ToT** | עץ עם pruning | k × עומק |
| **MCTS** | גרף כללי עם backpropagation | הרבה מאוד |

**ToT יקר** (הרבה קריאות LLM) אבל שימושי לבעיות שדורשות **חקירה** — תכנות, פאזלים, תכנון.

</div>

<div dir="rtl">

### למה דווקא "גרף"?

כשרואים את המילה "גרף" כאן — הכוונה לגרף מכוון (directed graph) מתורת הגרפים. אבל הניואנס החשוב: הגרף כאן הוא **מכונת מצבים** (state machine), לא מבנה נתונים.

| מושג מתמטי | תפקיד ב-LangGraph |
|------------|------------------|
| **צומת** (node) | פונקציה שמבצעת פעולה (קריאה ל-LLM, הרצת כלי...) |
| **קשת** (edge) | מעבר בין פעולות — יכולה להיות **מותנית** על ה-state |
| **מצב** (state) | מילון שמצטבר ועובר בין הצמתים — הזיכרון של הריצה |

**מה שמייחד את הגרף פה מ-DAG רגיל:** מותרת **מחזוריות** (cycles). הסוכן יכול לחזור לצומת שכבר ביקר בו — זה בדיוק מה שמאפשר לולאת חשיבה-פעולה-תצפית.

```
# DAG (LangChain chain רגיל) — אסורה חזרה:
A → B → C → END

# LangGraph — מותרת חזרה:
START → agent → tools → agent → tools → agent → END
                 ↑__________________________|
                 (חוזר כל עוד יש tool calls)
```

**ה-State הוא הדבר הכי חשוב להבין:**
```python
class AgentState(TypedDict):
    messages: list          # כל ההיסטוריה
    current_plan: str       # מה הסוכן מתכנן עכשיו
    tool_results: list      # מה קיבל עד כה
    step_count: int         # כמה צעדים עשה
```
כל צומת קורא מה-state ומחזיר עדכון חלקי — בדיוק כמו `reducer` ב-Redux.

</div>

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

# ה-State — מה הסוכן זוכר בין שלבים
class AgentState(TypedDict):
    # add_messages = הוסף הודעות חדשות, אל תחליף ישנות
    messages: Annotated[list, add_messages]

# צומת הסוכן — קריאה למודל
def agent_node(state: AgentState):
    result = model_with_tools.invoke(state["messages"])
    return {"messages": [result]}

# בניית הגרף
graph_builder = StateGraph(AgentState)

# הוסף צמתים
graph_builder.add_node("agent", agent_node)
graph_builder.add_node("tools", ToolNode(tools))  # ToolNode מריץ את הכלים אוטומטית

# הוסף קשתות
graph_builder.add_edge(START, "agent")  # תמיד מתחיל מה-agent

# tools_condition: אם המודל ביקש כלי → tools, אחרת → END
graph_builder.add_conditional_edges("agent", tools_condition)

# אחרי הרצת כלים — חזור ל-agent
graph_builder.add_edge("tools", "agent")

# קמפל
graph = graph_builder.compile()

print("✓ Graph compiled successfully")

In [ ]:
# הצג ויזואליזציה של הגרף (אם יש תמיכה)
try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Visualization not available: {e}")
    print("\nGraph structure:")
    print(graph.get_graph().draw_ascii())

In [ ]:
# הרץ את הסוכן!
result = graph.invoke({
    "messages": [HumanMessage(content="What's the weather in Tel Aviv and Jerusalem? And what's 15 * 7?")]
})

# הדפס את כל ההודעות שנצברו
print("=" * 50)
for msg in result["messages"]:
    role = msg.__class__.__name__
    print(f"\n[{role}]")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"  Tool calls: {[tc['name'] for tc in msg.tool_calls]}")
    if msg.content:
        print(f"  Content: {msg.content}")

<div dir="rtl">

### מה קרה כאן?

הסוכן רץ בלולאה:
1. **agent**: קיבל את השאלה, החליט לקרוא ל-`get_weather` ו-`multiply`
2. **tools**: ביצע את שתי הקריאות
3. **agent**: קיבל את התוצאות, ניסח תשובה סופית
4. **tools_condition**: זיהה שאין עוד tool calls → שלח ל-END

LangGraph ניהל את כל הלוגיקה הזו אוטומטית.

</div>

<div dir="rtl">

### Persistence — שמירת מצב בין שיחות

LangGraph תומך ב-checkpointing: שמירת כל state לאחסון (memory, SQLite, Redis...).
כך הסוכן יכול להמשיך שיחה מאוחר יותר.

</div>

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# גרף עם זיכרון פר-thread
checkpointer = MemorySaver()
persistent_graph = graph_builder.compile(checkpointer=checkpointer)

# thread_id = מזהה שיחה ייחודי
config = {"configurable": {"thread_id": "conversation-1"}}

# שלח הודעה ראשונה
r1 = persistent_graph.invoke(
    {"messages": [HumanMessage(content="My name is Ori. What's 10 + 5?")]},
    config=config
)
print("First response:", r1["messages"][-1].content)

# שלח הודעה שנייה — הסוכן זוכר את ההקשר!
r2 = persistent_graph.invoke(
    {"messages": [HumanMessage(content="What's my name? And double the result from before.")]},
    config=config
)
print("Second response:", r2["messages"][-1].content)

<div dir="rtl">

---

# חלק ג: MCP — Model Context Protocol

## הבעיה שMCP פותר

דמיין שיש לך 5 סוכנים שונים (LangGraph, CrewAI, AutoGen...) ו-20 כלים חיצוניים (GitHub, Notion, Slack, Jira...).

**ללא MCP:**
```
Agent1 ──→ GitHub integration (custom)
Agent1 ──→ Notion integration (custom)
Agent2 ──→ GitHub integration (different custom!)
Agent2 ──→ Slack integration (custom)
...5 agents × 20 tools = 100 integrations 😱
```

**עם MCP:**
```
Agent1 ──┐
Agent2 ──┤─→ MCP Protocol ─→ GitHub MCP Server
Agent3 ──┘                ─→ Notion MCP Server
                          ─→ Slack MCP Server
...5 agents + 20 servers = 25 integrations ✨
```

## מה MCP מגדיר?

MCP הוא **פרוטוקול תקשורת** (כמו HTTP, אבל לכלים של AI):

| מושג MCP | תיאור | דוגמה |
|----------|-------|-------|
| **Tools** | פונקציות שהסוכן יכול לקרוא | `read_file`, `create_issue` |
| **Resources** | קבצים/נתונים שהסוכן יכול לקרוא | `file://readme.md`, `db://users` |
| **Prompts** | תבניות prompt מוכנות | `code_review_template` |
| **Sampling** | הserver מבקש מה-LLM להשלים | (מתקדם) |

</div>

In [ ]:
# MCP Server פשוט — זה מה שספק הכלי כותב
# (הרץ את זה בטרמינל נפרד, או שמור לקובץ)

MCP_SERVER_CODE = '''
# weather_server.py — MCP server לפי הפרוטוקול
from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp.types import Tool, TextContent
import json

app = Server("weather-server")

@app.list_tools()
async def list_tools():
    return [
        Tool(
            name="get_weather",
            description="Get weather for a city",
            inputSchema={
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"]
            }
        )
    ]

@app.call_tool()
async def call_tool(name: str, arguments: dict):
    if name == "get_weather":
        city = arguments["city"]
        # בפועל — קריאה ל-API אמיתי
        result = f"Weather in {city}: 25°C, sunny"
        return [TextContent(type="text", text=result)]

if __name__ == "__main__":
    import asyncio
    asyncio.run(stdio_server(app))
'''

print("MCP Server example code:")
print(MCP_SERVER_CODE)

In [ ]:
# שמור את ה-server לקובץ להדגמה
server_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("demo-server")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@mcp.tool()
def get_weather(city: str) -> str:
    """Get weather for a city."""
    data = {"tel aviv": "25C sunny", "jerusalem": "18C cloudy"}
    return data.get(city.lower(), f"No data for {city}")

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("/tmp/demo_mcp_server.py", "w") as f:
    f.write(server_code)

print("✓ MCP server saved to /tmp/demo_mcp_server.py")

<div dir="rtl">

### חיבור MCP ל-LangGraph דרך langchain-mcp-adapters

הספרייה `langchain-mcp-adapters` עושה את הגשר: MCP tools → LangChain tools → LangGraph agent

</div>

In [ ]:
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

async def run_agent_with_mcp():
    # הגדר את ה-MCP servers שאנחנו רוצים להתחבר אליהם
    async with MultiServerMCPClient(
        {
            "demo": {
                "command": "python",
                "args": ["/tmp/demo_mcp_server.py"],
                "transport": "stdio",
            }
        }
    ) as client:
        # המרת MCP tools → LangChain tools (אוטומטית!)
        mcp_tools = client.get_tools()
        print(f"Tools loaded from MCP: {[t.name for t in mcp_tools]}")
        
        # create_react_agent = קיצור דרך ליצירת סוכן ReAct עם LangGraph
        agent = create_react_agent(model, mcp_tools)
        
        result = await agent.ainvoke({
            "messages": [HumanMessage(content="What's the weather in Tel Aviv? Also compute 42 + 58.")]
        })
        
        print("\nFinal answer:")
        print(result["messages"][-1].content)

# הרץ
await run_agent_with_mcp()

<div dir="rtl">

---

# חלק ד: כולם יחד — Big Picture

```
┌─────────────────────────────────────────────────────┐
│                   Your Application                  │
│                                                     │
│  ┌──────────────────────────────────────────────┐   │
│  │              LangGraph (orchestration)       │   │
│  │                                              │   │
│  │   START → [agent] ⇄ [tools] → END           │   │
│  │              │                               │   │
│  │   ┌──────────┴──────────────────────────┐   │   │
│  │   │         LangChain                   │   │   │
│  │   │  ChatAnthropic + PromptTemplate      │   │   │
│  │   │  + Memory + Tool binding             │   │   │
│  │   └──────────┬──────────────────────────┘   │   │
│  └──────────────┼───────────────────────────────┘   │
│                 │                                   │
│  ┌──────────────▼───────────────────────────────┐   │
│  │              MCP Layer                       │   │
│  │   langchain-mcp-adapters                     │   │
│  └──────┬────────────┬───────────┬──────────────┘   │
│         │            │           │                  │
│    [GitHub     [Notion      [Your custom             │
│     server]     server]      server]                 │
└─────────────────────────────────────────────────────┘
```

## סיכום — מתי להשתמש בכל אחד?

### LangChain לבד:
- שאילתות פשוטות ל-LLM
- Pipeline חד-כיווני (input → process → output)
- אין צורך בלולאות

### LangGraph (+ LangChain):
- **סוכנים** שצריכים להחליט ולפעול
- **לולאות** (המודל קורא לכלי, מקבל תוצאה, ממשיך)
- **מצב** שמצטבר לאורך השיחה
- **Human-in-the-loop** — עצירה לאישור אנושי
- **Multi-agent** — כמה סוכנים שמתקשרים

### MCP (+ כל השאר):
- כשאתה **ספק** כלים שסוכנים אחרים ישתמשו בהם
- כשאתה **צורך** כלים מספקים חיצוניים (GitHub, Notion...)
- כשאתה רוצה **סטנדרט** שעובד עם כל framework

</div>

<div dir="rtl">

---

# בונוס: סוכן מורכב יותר — Human-in-the-Loop

אחד הכוחות הגדולים של LangGraph: `interrupt_before` — עצירה לפני ביצוע פעולה לאישור אנושי.

</div>

In [ ]:
from langgraph.types import interrupt

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email."""
    # בפועל — SMTP או API
    return f"Email sent to {to}: '{subject}'"

@tool
def delete_file(path: str) -> str:
    """Delete a file from disk."""
    return f"File deleted: {path}"

# סוכן עם interrupt לפני פעולות מסוכנות
dangerous_tools = [send_email, delete_file]
safe_tools = [add, get_weather]

model_with_all_tools = model.bind_tools(dangerous_tools + safe_tools)

def agent_with_approval(state: AgentState):
    result = model_with_all_tools.invoke(state["messages"])
    
    # בדוק אם יש כלי מסוכן
    if result.tool_calls:
        dangerous_names = {t.name for t in dangerous_tools}
        for tc in result.tool_calls:
            if tc["name"] in dangerous_names:
                # עצור ובקש אישור!
                approval = interrupt({
                    "question": f"Approve calling '{tc['name']}' with {tc['args']}?",
                    "tool_call": tc
                })
                if not approval:
                    return {"messages": [{"role": "assistant", "content": "Action cancelled by user."}]}
    
    return {"messages": [result]}

print("✓ Human-in-the-loop agent defined")
print("   → Will pause and ask for approval before send_email or delete_file")

<div dir="rtl">

---

# משאבים להמשך

### תיעוד רשמי
- [LangChain docs](https://python.langchain.com/docs/)
- [LangGraph docs](https://langchain-ai.github.io/langgraph/)
- [MCP spec](https://modelcontextprotocol.io/)
- [langchain-mcp-adapters](https://github.com/langchain-ai/langchain-mcp-adapters)

### MCP Servers מוכנים
- [GitHub MCP Server](https://github.com/github/github-mcp-server)
- [Filesystem MCP](https://github.com/modelcontextprotocol/servers/tree/main/src/filesystem)
- [MCP Servers Hub](https://mcp.so/)

### LangGraph patterns מתקדמים
- Multi-agent networks
- Subgraphs
- LangGraph Platform (cloud deployment)
- LangSmith (observability)

</div>